In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, warnings
import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

PROJECT = '/content/drive/MyDrive/ages-sprint-overrun/pipeline'
IN  = f'{PROJECT}/01_data_prep'
OUT = f'{PROJECT}/02_benchmark'
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv(f'{IN}/df_final.csv', parse_dates=['Created At'])
FEATURE_COLS = joblib.load(f'{IN}/feature_cols.pkl')

print(f'Loaded: {len(df)} rows, {len(FEATURE_COLS)} features')
print(f'Exceed-sprint rate: {df["depaseste_sprint"].mean()*100:.1f}%')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: 13538 rows, 27 features
Exceed-sprint rate: 38.4%


In [ ]:
# Created chronological split (80/20)

df = df.sort_values('Created At').reset_index(drop=True)
X = df[FEATURE_COLS].copy()
y = df['depaseste_sprint'].copy()

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

print(f'Train: {len(X_train)}  ({df["Created At"].iloc[0].date()} -> '
      f'{df["Created At"].iloc[split_idx-1].date()})')
print(f'Test:  {len(X_test)}  ({df["Created At"].iloc[split_idx].date()} -> '
      f'{df["Created At"].iloc[-1].date()})')
print(f'\nTrain exceed-sprint rate: {y_train.mean()*100:.1f}%')
print(f'Test exceed-sprint rate:  {y_test.mean()*100:.1f}%')

split_info = pd.DataFrame([
    {'Set': 'Train', 'N': len(X_train), 'Exceed-sprint (%)': round(100*y_train.mean(),1)},
    {'Set': 'Test',  'N': len(X_test),  'Exceed-sprint (%)': round(100*y_test.mean(),1)},
])
split_info.to_csv(f'{OUT}/table_split_info.csv', index=False)
split_info

Train: 10830  (2011-09-16 -> 2023-12-08)
Test:  2708  (2023-12-08 -> 2024-05-11)

Train exceed-sprint rate: 39.9%
Test exceed-sprint rate:  32.7%


,Set,N,Exceed-sprint (%)
0,Train,10830,39.9
1,Test,2708,32.7


In [ ]:
# Internal validation scheme - time series split

from sklearn.model_selection import TimeSeriesSplit

N_SPLITS = 10
cv_ts = TimeSeriesSplit(n_splits=N_SPLITS)

for i, (tr, va) in enumerate(cv_ts.split(X_train), 1):
    print(f'Fold {i:2d}: train={len(tr):6d}  val={len(va):5d}  '
          f'val exceed-rate={y_train.iloc[va].mean()*100:.1f}%')

Fold  1: train=   990  val=  984  val exceed-rate=35.0%
Fold  2: train=  1974  val=  984  val exceed-rate=29.3%
Fold  3: train=  2958  val=  984  val exceed-rate=55.1%
Fold  4: train=  3942  val=  984  val exceed-rate=48.5%
Fold  5: train=  4926  val=  984  val exceed-rate=40.1%
Fold  6: train=  5910  val=  984  val exceed-rate=39.4%
Fold  7: train=  6894  val=  984  val exceed-rate=41.0%
Fold  8: train=  7878  val=  984  val exceed-rate=43.9%
Fold  9: train=  8862  val=  984  val exceed-rate=36.6%
Fold 10: train=  9846  val=  984  val exceed-rate=35.7%


In [ ]:
# Benchmarok - 7 baseline models
# SMOTE is applied inside each training fold, never on validation/test data

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier)
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                             precision_score, recall_score, matthews_corrcoef)

def pipe(clf, scale=False):
    steps = [('smote', SMOTE(random_state=RANDOM_STATE))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('clf', clf))
    return ImbPipeline(steps)

modele_base = {
    'Logistic Regression': pipe(LogisticRegression(max_iter=1000,
                                random_state=RANDOM_STATE), scale=True),
    'Decision Tree': pipe(DecisionTreeClassifier(max_depth=5,
                          random_state=RANDOM_STATE)),
    'Random Forest': pipe(RandomForestClassifier(n_estimators=200, max_depth=8,
                          random_state=RANDOM_STATE, n_jobs=-1)),
    'Gradient Boosting': pipe(GradientBoostingClassifier(n_estimators=200, max_depth=4,
                              learning_rate=0.05, random_state=RANDOM_STATE)),
    'AdaBoost': pipe(AdaBoostClassifier(n_estimators=100, learning_rate=0.1,
                     random_state=RANDOM_STATE)),
    'XGBoost': pipe(XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
                    n_jobs=-1, eval_metric='logloss')),
    'MLP': pipe(MLPClassifier(hidden_layer_sizes=(100, 50), activation='relu',
                max_iter=500, early_stopping=True, n_iter_no_change=10,
                random_state=RANDOM_STATE), scale=True),
}

rezultate_base = {}
rows = []

for nume, model in modele_base.items():
    t0 = time.time()
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv_ts,
                                scoring='roc_auc', n_jobs=-1)
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    dt = time.time() - t0

    rezultate_base[nume] = {'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
                            'cv_scores': cv_scores}

    rows.append({
        'Model': nume,
        'CV AUC': round(cv_scores.mean(), 3),
        'CV AUC (SD)': round(cv_scores.std(), 3),
        'Test AUC': round(roc_auc_score(y_test, y_proba), 3),

        'Precision (Exceeds)': round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 3),
        'Recall (Exceeds)': round(recall_score(y_test, y_pred, pos_label=1), 3),
        'F1 (Exceeds)': round(f1_score(y_test, y_pred, pos_label=1), 3),
        'MCC': round(matthews_corrcoef(y_test, y_pred), 3),

        'Accuracy (weighted)': round(accuracy_score(y_test, y_pred), 3),
        'Precision (weighted)': round(precision_score(y_test, y_pred, average='weighted'), 3),
        'Recall (weighted)': round(recall_score(y_test, y_pred, average='weighted'), 3),
        'F1 (weighted)': round(f1_score(y_test, y_pred, average='weighted'), 3),
        'Time (s)': round(dt, 1),
    })
    print(f'{nume:22s} CV AUC={cv_scores.mean():.3f} ± {cv_scores.std():.3f}  ({dt:.1f}s)')

df_base = pd.DataFrame(rows).sort_values('MCC', ascending=False).reset_index(drop=True)

Logistic Regression    CV AUC=0.611 ± 0.049  (3.7s)
Decision Tree          CV AUC=0.598 ± 0.040  (1.6s)
Random Forest          CV AUC=0.642 ± 0.043  (18.8s)
Gradient Boosting      CV AUC=0.636 ± 0.045  (48.4s)
AdaBoost               CV AUC=0.617 ± 0.047  (8.6s)
XGBoost                CV AUC=0.624 ± 0.045  (8.6s)
MLP                    CV AUC=0.614 ± 0.048  (20.2s)


In [ ]:
# Calculation of Precision, Recall, F1 and MCC on the exceed-sprint class

cols_per_class = ['Model', 'CV AUC', 'CV AUC (SD)', 'Test AUC',
                   'Precision (Exceeds)', 'Recall (Exceeds)', 'F1 (Exceeds)', 'MCC']
cols_weighted  = ['Model', 'Accuracy (weighted)', 'Precision (weighted)',
                   'Recall (weighted)', 'F1 (weighted)']

df_base[cols_per_class].to_csv(f'{OUT}/table_baseline_per_class.csv', index=False)
df_base[cols_weighted].to_csv(f'{OUT}/table_baseline_weighted.csv', index=False)
df_base.to_csv(f'{OUT}/table_baseline_full.csv', index=False)

print('PRIMARY (per-class):')
print(df_base[cols_per_class].to_string(index=False))
print()
print('SECONDARY (weighted):')
print(df_base[cols_weighted].to_string(index=False))

PRIMARY (per-class):
              Model  CV AUC  CV AUC (SD)  Test AUC  Precision (Exceeds)  Recall (Exceeds)  F1 (Exceeds)   MCC
  Gradient Boosting   0.636        0.045     0.694                0.519             0.518         0.518 0.284
      Random Forest   0.642        0.043     0.693                0.508             0.521         0.515 0.274
           AdaBoost   0.617        0.047     0.695                0.506             0.505         0.505 0.265
            XGBoost   0.624        0.045     0.684                0.502             0.481         0.491 0.252
      Decision Tree   0.598        0.040     0.650                0.454             0.499         0.475 0.202
Logistic Regression   0.611        0.049     0.623                0.452             0.421         0.436 0.176
                MLP   0.614        0.048     0.627                0.443             0.442         0.443 0.172

SECONDARY (weighted):
              Model  Accuracy (weighted)  Precision (weighted)  Recall (weig

## Save split, CV scheme, and trained baseline models

In [ ]:
# Save results

joblib.dump((X_train, X_test, y_train, y_test), f'{OUT}/split_chronological.pkl')
joblib.dump(cv_ts, f'{OUT}/cv_scheme.pkl')
joblib.dump(rezultate_base, f'{OUT}/rezultate_base.pkl')

print(f'Saved to {OUT}')

Saved to /content/drive/MyDrive/ages-sprint-overrun/pipeline/02_benchmark
